# 50-70 Years Old Weather Stations - Temperature Anomalies
This notebook calculates the overall average `Max_Temp_C` for stations with a history between 50 and 70 years, fully present until the end of the dataset. It then computes the average `Max_Temp_C` for each year, and visualizes the difference (anomaly) as a bar diagram. Red bars indicate warmer-than-average years, and blue bars indicate cooler-than-average years.

In [ ]:
import pyarrow.dataset as ds
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Load dataset
dataset = ds.dataset('../data/dwd_historical_lake', format='parquet', partitioning='hive')

# Find stations with 50-70 years of history that reach the end of the dataset
df_meta = dataset.to_table(columns=['Station_ID', 'Date']).to_pandas()
df_meta['Date'] = pd.to_datetime(df_meta['Date'])

station_stats = df_meta.groupby('Station_ID')['Date'].agg(['max', 'count'])
station_stats['years'] = station_stats['count'] / 365.25
filtered = station_stats[
    (station_stats['years'] >= 50) & 
    (station_stats['years'] <= 70) & 
    (station_stats['max'] >= pd.Timestamp('2024-12-31'))
]
stations_50_70 = filtered.index.tolist()

print(f"Processing anomalies for {len(stations_50_70)} stations with 50-70 years of history...")

import io
import requests

print("Fetching station names...")
url = 'https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt'
response = requests.get(url)
response.encoding = 'latin-1'
col_specs = [(0, 5), (6, 14), (15, 23), (24, 38), (39, 50), (51, 60), (61, 102), (102, 140)]
col_names = ["Station_ID", "Start_Date", "End_Date", "Altitude", "Latitude", "Longitude", "Station_Name", "Bundesland"]
df_meta_names = pd.read_fwf(io.StringIO(response.text), skiprows=2, names=col_names, colspecs=col_specs, dtype={"Station_ID": str})
df_meta_names['Station_ID'] = df_meta_names['Station_ID'].str.zfill(5)
df_meta_names['Station_Name'] = df_meta_names['Station_Name'].str.strip()
name_dict = df_meta_names.set_index('Station_ID')['Station_Name'].to_dict()


In [ ]:
# Load Max_Temp_C and metadata for those stations
df_temp = dataset.to_table(columns=['Station_ID', 'Date', 'Max_Temp_C', 'Berechnungs_Methode', 'Sensor_Typ', 'Strahlungsschutz']).to_pandas()
df_50_70 = df_temp[df_temp['Station_ID'].isin(stations_50_70)].copy()

# Add Year column
df_50_70['Year'] = df_50_70['Date'].dt.year

# 1. Calculate overall average max_temp per station
overall_avg = df_50_70.groupby('Station_ID')['Max_Temp_C'].mean().reset_index()
overall_avg.rename(columns={'Max_Temp_C': 'Overall_Avg'}, inplace=True)

# 2. Calculate yearly average max_temp per station
yearly_avg = df_50_70.groupby(['Station_ID', 'Year'])['Max_Temp_C'].mean().reset_index()

# 3. Merge and calculate anomaly
anomaly_df = pd.merge(yearly_avg, overall_avg, on='Station_ID')
anomaly_df['Anomaly'] = anomaly_df['Max_Temp_C'] - anomaly_df['Overall_Avg']

# Plotting loop
for station in stations_50_70:
    name = name_dict.get(station, 'Unknown')
    station_data = anomaly_df[anomaly_df['Station_ID'] == station]
    station_raw = df_50_70[df_50_70['Station_ID'] == station].sort_values('Date')
    
    plt.figure(figsize=(12, 5))
    
    # Create colors: red for positive anomaly (warmer), blue for negative (cooler)
    colors = ['#d73027' if x > 0 else '#4575b4' for x in station_data['Anomaly']]
    
    plt.bar(station_data['Year'], station_data['Anomaly'], color=colors, width=1.0)
    
    # Adding baseline line
    plt.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
    
    plt.title(f'Yearly Max Temp Anomaly for {name} ({station})', fontsize=14, fontweight='bold')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Anomaly (°C) from Baseline', fontsize=12)
    
    # Plot metadata changes
    changes = []
    for col in ['Berechnungs_Methode', 'Sensor_Typ', 'Strahlungsschutz']:
        mask = (station_raw[col] != station_raw[col].shift(1)) & (station_raw[col].notna())
        if len(mask) > 0:
            mask.iloc[0] = False
        changed_rows = station_raw[mask]
        for _, row in changed_rows.iterrows():
            changes.append({'date': row['Date'], 'col': col, 'val': row[col]})
            
    y_min, y_max = plt.ylim()
    y_range = y_max - y_min
    for i, c in enumerate(changes):
        x_pos = c['date'].year + c['date'].dayofyear / 365.25
        plt.axvline(x=x_pos, color='gray', linestyle='--', alpha=0.7)
        short_val = str(c['val'])[:25] + ('...' if len(str(c['val'])) > 25 else '')
        y_text_pos = y_min + y_range * (0.05 + (i % 3) * 0.15)
        plt.text(x_pos, y_text_pos, f"{c['col']}:\n{short_val}", rotation=90, 
                 verticalalignment='bottom', fontsize=8,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6))
    
    # Add text box for the baseline temperature
    baseline_temp = station_data['Overall_Avg'].iloc[0]
    plt.text(0.01, 0.95, f'Baseline (Historical Avg Max Temp): {baseline_temp:.2f}°C', 
             transform=plt.gca().transAxes, fontsize=11, verticalalignment='top', 
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))
             
    plt.tight_layout()
    plt.show()

